# 02 — LoRA SFT training (Stage S3, Colab T4)

Fine-tune **Llama-3.2-1B-Instruct** with LoRA on MIDI-Instruct edit shards.

**Self-contained:** this notebook bootstraps a fresh Colab runtime (clone, install, restore/build shards). Keep `RUN_ID` and `USE_DRIVE` aligned with notebook **01**.

In [ ]:
# @title Configuration (must match notebook 01)
MIDI_LLM_REPO = "https://github.com/juliagsy/midi-llm.git"
MUSICINSTRUCT_REPO = "https://github.com/juliagsy/musicinstruct.git"
BRANCH = "main"

WORK = "/content"
MIDI_LLM = f"{WORK}/midi-llm"
MUSICINSTRUCT = f"{WORK}/musicinstruct"
RUN_ID = "pilot_v1"
RUNS_ROOT = f"{WORK}/runs/{RUN_ID}"
SHARDS_ROOT = f"{WORK}/data/shards/{RUN_ID}"

# Training knobs — T4 defaults
REPR = "remi"  # @param ["remi", "octuple", "amt"]
TRAIN_ALL_REPRS = False  # @param {type:"boolean"}
MAX_STEPS = 300  # @param {type:"integer"}
MAX_SEQ_LEN = 1024  # @param {type:"integer"}  # T4 override; protocol default is 2048 (configs/base.yaml)
MAX_SAMPLES = None  # @param {type:"raw"}  # None = full train shard
BATCH_SIZE = 2  # @param {type:"integer"}
GRAD_ACCUM = None  # @param {type:"raw"}  # None → derive from configs/base.yaml effective_batch_tokens
LEARNING_RATE = 2e-4  # @param {type:"number"}
PRECISION = "fp16"  # @param ["fp16", "bf16"]  T4 → fp16
SEED = 42  # @param {type:"integer"}

USE_DRIVE = True  # @param {type:"boolean"}  recommended: each notebook gets a new runtime
DRIVE_ROOT = "/content/drive/MyDrive/midi-llm"
DRIVE_SHARDS = f"{DRIVE_ROOT}/shards/{RUN_ID}"
DRIVE_RUNS = f"{DRIVE_ROOT}/runs/{RUN_ID}"

In [ ]:
# @title Mount Google Drive (checkpoints + shard restore)
import os

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RUNS_ROOT = DRIVE_RUNS
    os.makedirs(RUNS_ROOT, exist_ok=True)
    print("Checkpoints on Drive:", RUNS_ROOT)
    print("Shard restore path:", DRIVE_SHARDS)
else:
    print("Ephemeral VM paths — shards:", SHARDS_ROOT, "| runs:", RUNS_ROOT)

In [ ]:
# @title Bootstrap session (clone, install, restore or build shards)
import os
import shutil
import subprocess
from pathlib import Path

def _sh(cmd: str) -> None:
    subprocess.run(cmd, shell=True, check=True)

if not Path(MIDI_LLM).is_dir():
    _sh(f"git clone --branch {BRANCH} --depth 1 {MIDI_LLM_REPO} {MIDI_LLM}")
if not Path(MUSICINSTRUCT).is_dir():
    _sh(f"git clone --branch {BRANCH} --depth 1 {MUSICINSTRUCT_REPO} {MUSICINSTRUCT}")

%cd {MIDI_LLM}
# Colab preinstalls torchao 0.10; peft requires >=0.16
!pip install -q -U "torchao>=0.16.0" || pip uninstall -y torchao
!pip install -q -e ".[repr,train]"
!pip install -q "git+https://github.com/jthickstun/anticipation.git" mido

if not os.environ.get("HF_TOKEN"):
    from getpass import getpass
    from huggingface_hub import login
    os.environ["HF_TOKEN"] = getpass("HF token (hf_...): ")
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

manifest = Path(MUSICINSTRUCT) / "data/pilot/pilot.jsonl"
shards_dir = Path(SHARDS_ROOT)
shard_ok = shards_dir.is_dir() and any(shards_dir.glob("edit_*_train.jsonl"))

if not shard_ok and USE_DRIVE and Path(DRIVE_SHARDS).is_dir():
    shards_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(DRIVE_SHARDS, SHARDS_ROOT, dirs_exist_ok=True)
    shard_ok = any(shards_dir.glob("edit_*_train.jsonl"))
    if shard_ok:
        print("Restored shards from Drive:", DRIVE_SHARDS)

if not shard_ok:
    print("Building shards (first run or no Drive copy)...")
    if not manifest.is_file():
        !cd {MUSICINSTRUCT} && musicinstruct generate-pilot --output-dir data/pilot --target 300
    os.makedirs(SHARDS_ROOT, exist_ok=True)
    for split in ("train", "validation"):
        for repr_name in ("remi", "octuple", "amt"):
            out = f"{SHARDS_ROOT}/edit_{repr_name}_{split}.jsonl"
            !midi-llm build-instruct-shard {manifest} --repr {repr_name} --output {out} --split {split}
    if USE_DRIVE:
        Path(DRIVE_SHARDS).parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(SHARDS_ROOT, DRIVE_SHARDS, dirs_exist_ok=True)
        print("Synced new shards to Drive:", DRIVE_SHARDS)

print("Shard files:", sorted(p.name for p in shards_dir.glob("*.jsonl")))

In [ ]:
# @title GPU check
import torch
assert torch.cuda.is_available(), "Enable T4 GPU runtime"
print(torch.cuda.get_device_name(0))

In [ ]:
# @title Train LoRA
from pathlib import Path
from midi_llm.config import load_config
from midi_llm.config_helpers import resolve_grad_accum_steps
from midi_llm.train.lora_sft import train_lora_sft

reprs = ["remi", "octuple", "amt"] if TRAIN_ALL_REPRS else [REPR]
results = {}
for repr_name in reprs:
    shard = Path(SHARDS_ROOT) / f"edit_{repr_name}_train.jsonl"
    if not shard.is_file():
        raise FileNotFoundError(f"Missing shard {shard} — re-run bootstrap cell above")
    out_dir = Path(RUNS_ROOT) / repr_name
    cfg = load_config(repr_name)
    protocol_seq = cfg["model"]["max_seq_len"]
    if MAX_SEQ_LEN != protocol_seq:
        print(f"Note: MAX_SEQ_LEN={MAX_SEQ_LEN}; paper protocol is {protocol_seq}")
    grad_accum = GRAD_ACCUM
    if grad_accum is None:
        grad_accum = resolve_grad_accum_steps(
            cfg, seq_len=MAX_SEQ_LEN, batch_size=BATCH_SIZE
        )
    print(f"\n=== Training {repr_name} → {out_dir} (grad_accum={grad_accum}) ===")
    train_lora_sft(
        shard,
        out_dir,
        repr_name=repr_name,
        max_steps=MAX_STEPS,
        max_seq_len=MAX_SEQ_LEN,
        max_samples=MAX_SAMPLES,
        per_device_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=grad_accum,
        learning_rate=LEARNING_RATE,
        precision=PRECISION,
        seed=SEED,
        logging_steps=10,
        save_steps=min(200, max(50, MAX_STEPS // 2)),
    )
    results[repr_name] = str(out_dir / "lora_adapter")
    print("Saved:", results[repr_name])

print("\nAdapters:", results)

In [ ]:
# @title Sync checkpoints to Drive + optional zip download
import shutil
from pathlib import Path

if USE_DRIVE:
    Path(DRIVE_RUNS).parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(RUNS_ROOT, DRIVE_RUNS, dirs_exist_ok=True)
    print("Synced checkpoints to Drive:", DRIVE_RUNS)

zip_path = f"{WORK}/runs_{RUN_ID}_adapters"
shutil.make_archive(zip_path, "zip", RUNS_ROOT)
print("Local zip:", zip_path + ".zip")
try:
    from google.colab import files
    files.download(zip_path + ".zip")
except ImportError:
    pass

## T4 timing guide

| `MAX_STEPS` | `MAX_SEQ_LEN` | ~Time (1 arm) |
|-------------|---------------|---------------|
| 50 | 512 | 5–10 min |
| 300 | 1024 | 30–45 min |
| 3000 | 1024 | 4–6 h |

If OOM: set `BATCH_SIZE=1`, `MAX_SEQ_LEN=512`, `GRAD_ACCUM=8`.

## Next

1. Confirm **Sync checkpoints to Drive** ran (if `USE_DRIVE=True`).
2. Disconnect this runtime (optional).
3. Open **`03_eval_musicinstruct.ipynb`** in a **new** session (same `RUN_ID`, `USE_DRIVE=True`).